# Categorize Responses in the Behavioural Set

In [ ]:
import sys, os

PARENT_DIR = os.path.abspath(os.path.join(os.getcwd(),".."))
if PARENT_DIR not in sys.path:
    sys.path.insert(0,PARENT_DIR)

In [ ]:
import torch
import json
from typing import List, Literal, Optional, Union, Dict
from enum import Enum
import datetime
from dataclasses import dataclass
from pydantic import BaseModel, Field
from transformers import AutoTokenizer, AutoModelForTokenClassification
from config import settings
from gcp_utils import download_from_gcs
from inference.v01.inference_utils import predict_word_level, word_labels_to_spans

In [ ]:
@dataclass 
class ErrorTaxonomy: 
    type_0 : str = "Irrelevant Span Mislabeling"
    type_1: str = "Missing Expected Entity"
    type_2: str = "Tokenization Artifacts"
    type_3: str = "Incorrect Polarity Assignment"
    type_4: str = "BIO Sequencing Errors"
    type_5: str = "Boundary Overreach"
    type_6: str = "Boundary Undereach"
    type_7: str = "Model Overgeneralization"

ERROR_TAXONOMY = {
    "0":ErrorTaxonomy.type_0,
    "1":ErrorTaxonomy.type_1,
    "2":ErrorTaxonomy.type_2,
    "3":ErrorTaxonomy.type_3,
    "4":ErrorTaxonomy.type_4,
    "5":ErrorTaxonomy.type_5,
    "6":ErrorTaxonomy.type_6,
    "7":ErrorTaxonomy.type_7
}

MODEL_NAME= "dmis-lab/biobert-base-cased-v1.1"
RUN_IDX ="2"# BEST RUN
VERSION = "v01"
# inference pipeline version may change as we improve and modify the pipeline
INFERENCE_PIPELINE_VERSION = "v01" 

# DATACLASSES TO DEFINE LOG ENTRY
@dataclass
class Spans:
    start: int
    end: int
    text: str
    label: str  #Enum['O', 'SYMPTOM_POS', 'SYMPTOM_NEG', f'CONFLICT-{some variable}']
@dataclass
class Reviewer(Enum):
    HUMAN = "HUMAN"
    AI = "AI"
    CODE = "CODE"
@dataclass
class LogEntry:
    date: str
    reviewer: Reviewer  # Only Reviewer.HUMAN or Reviewer.AI
    input_text: str
    tokens: List[str]
    token_level_labels: List[str]
    word_level_labels: List[str]  
    predicted_spans: List[Spans]
    expected_entities: List[Spans]
    error_type: Optional[List[ErrorTaxonomy]| ErrorTaxonomy]
    reasoning: str 
    model: str = MODEL_NAME
    model_version: str = VERSION
    inference_pipeline_version: str = INFERENCE_PIPELINE_VERSION

## **Load Required Data Files**

In [ ]:
## Load Required Data Files

# Load id2label mapping
print("📂 Loading id2label mapping...")
with open("data/id2label.json", "r") as f:
    id2label = json.load(f)

# Convert id2label keys from strings to integers (JSON loads keys as strings)
if any(isinstance(k, str) for k in id2label.keys()):
    id2label = {int(k): v for k, v in id2label.items()}

print(f"✅ Loaded {len(id2label)} label mappings")
print(f"Labels: {id2label}")

# Load behavioural evaluation set
print("\n📂 Loading behavioural evaluation set...")
with open("behavioural_set.json", "r") as f:
    behavioural_set = json.load(f)

print(f"✅ Loaded {len(behavioural_set)} categories")
print(f"Categories: {list(behavioural_set.keys())}")

# Count total examples
total_examples = sum(len(examples) for examples in behavioural_set.values())
print(f"Total test examples: {total_examples}")

## **Load model from local directory or GCS**

In [ ]:
# Load the model
GCS_MODEL_PATH = f"{VERSION}/runs/{MODEL_NAME}/run_{RUN_IDX}"
BUCKET_NAME = settings.BUCKET_NAME  # "ner_training_data_results"

# Create a local directory path for the model
LOCAL_MODEL_DIR = f"./downloaded_models/{MODEL_NAME}/run_{RUN_IDX}"

# Check if model already exists locally
if os.path.exists(LOCAL_MODEL_DIR) and os.path.isfile(os.path.join(LOCAL_MODEL_DIR, "config.json")):
    print(f"✅ Model found locally at {LOCAL_MODEL_DIR}")
    print("Skipping download from GCS.")
else:
    # Download the model directory from GCS if not found locally
    print(f"📥 Model not found locally. Downloading from gs://{BUCKET_NAME}/{GCS_MODEL_PATH}...")
    downloaded_path = download_from_gcs(
        gcs_path=GCS_MODEL_PATH,
        local_path=LOCAL_MODEL_DIR,
        bucket_name=BUCKET_NAME
    )
    if downloaded_path:
        print(f"✅ Download complete. Model saved to {LOCAL_MODEL_DIR}")

# Load the model
print(f"📂 Loading model from {LOCAL_MODEL_DIR}...")
model = AutoModelForTokenClassification.from_pretrained(LOCAL_MODEL_DIR)
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_DIR)

# Move to device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
    
print(f"Using device: {device}")
model.to(device)
model.eval()
print("✅ Model is ready to be used")

## EVALUATION LOOP

In [ ]:
@dataclass
class BehaviouralExample:
    example: str
    entities_with_labels: List[Spans]


In [ ]:
# Fit the behavioural set into the dataclasses
for k,examples in behavioural_set.items():
    for i,ex in enumerate(examples):
        behavioural_set[k][i] = BehaviouralExample(**ex)

**HELPERS**

In [ ]:
# Extract spans from the model's prediction
# Check if entity text is contained within any span text (substring match)
# This handles cases where model predicted longer spans (e.g., "fever or chills" contains "fever" and "chills")
def _is_entity_in_spans(entity_text, spans):
    """
    Check if entity text is contained within any span text and return the span index.
    
    Returns:
        int | None: The index of the first span containing the entity text, or None if not found.
    
    How it works:
    - Uses `enumerate()` to get both the index and span from the spans list
    - For each span, checks if entity_text is a substring of span['text']
    - Returns the index as soon as a match is found (short-circuit evaluation)
    - Returns None if no match is found
    
    Example:
        entity_text = "fever"
        spans = [
            {'text': 'The patient', ...},      # index 0: "fever" not in "The patient" -> continue
            {'text': 'fever or chills', ...},  # index 1: "fever" in "fever or chills" -> return 1
            {'text': 'reports', ...}           # (not reached due to short-circuit)
        ]
        # Returns: 1
    """
    for idx, span in enumerate(spans):
        if entity_text in span['text']:
            return idx
    return None


def _check_boundaries(expected_start, expected_end, predicted_start, predicted_end):
    """
    Check for boundary errors (overreach/undereach).
    
    Returns:
        None: No boundary error
        ErrorTaxonomy.type_5: Overreach (predicted span is longer)
        ErrorTaxonomy.type_6: Undereach (predicted span is shorter)
    """
    # Calculate overlap
    overlap_start = max(expected_start, predicted_start)
    overlap_end = min(expected_end, predicted_end)
    overlap_length = max(0, overlap_end - overlap_start)
    
    expected_length = expected_end - expected_start
    predicted_length = predicted_end - predicted_start
    
    # If no meaningful overlap, this isn't a boundary error
    # (it's a different entity or wrong detection)
    if overlap_length < expected_length * 0.5:  # Less than 50% overlap
        # TODO: figure out what kind of error to return
        return None  # Not a boundary issue, likely type_0 or type_7 
    
    # Perfect match
    if expected_start == predicted_start and expected_end == predicted_end:
        return None
    
    # Overreach: predicted span extends beyond expected boundaries
    if predicted_start < expected_start or predicted_end > expected_end:
        return ErrorTaxonomy.type_5
    
    # Undereach: predicted span is contained within expected boundaries
    if predicted_start >= expected_start and predicted_end <= expected_end:
        return ErrorTaxonomy.type_6
    
    # Mixed case (starts before but ends early, or starts late but ends after)
    # This is tricky - could be either, but usually overreach is more common
    if predicted_length > expected_length:
        return ErrorTaxonomy.type_5
    else:
        return ErrorTaxonomy.type_6


def _check_predicted_labels(predicted_label:str, label: str):
    """this goes inside the if block for the detected entities, so we should not see"""
    print(f"LABEL ({type(label)}): {label}")
    print(f"PREDICTED LABEL ({type(predicted_label)}): {predicted_label}")
    if predicted_label == label:
        # This will catch the cases where label = "O", 
        # we included some tricky cases where symptoms are present in the cases but NOT in the context of a patient describing the symptoms that she/he does not have.
        return None
    if label == 'O' and predicted_label != 'O':
        # tricky case, where we did not want the model to detect this term as an entity
        # If we fall here then the model detected an irrelevant entity

        # check if there was a conflict:
        if predicted_label.startswith("CONFLICT"):
            return [ErrorTaxonomy.type_0, ErrorTaxonomy.type_2, ErrorTaxonomy.type_4]
        
        return ErrorTaxonomy.type_0
        
    if label != 'O' and predicted_label == 'O':
        return ErrorTaxonomy.type_1 # missed an entity

    if predicted_label.startswith("CONFLICT"):
        return [ErrorTaxonomy.type_2, ErrorTaxonomy.type_4]
    if predicted_label.split("_")[1] != label.split("_")[1]:
        return ErrorTaxonomy.type_3 # incorrect polarity assignment



In [ ]:
print("Working example:")
my_example = behavioural_set['Long, realistic clinical sentences (THIS IS GOLD 🥇)'][0]
text = my_example.example
print(f"\t{text}")
tokens, token_labels, word_ids, words, word_labels, word_offsets = predict_word_level(
        text=text,
        model=model,
        tokenizer=tokenizer,
        id2label=id2label,
        device=device,
    )
spans = word_labels_to_spans(text=text, word_offsets=word_offsets, word_labels=word_labels)
print("SPANS:")
for s in spans:
    print(f"\t{s}")

*Snipped to check if the expected entities are in the returned spans*

In [ ]:

# See which entities from the behavioural examples are in:
# For present entities, we also track which span index they were found in
present_with_indices = [(e, _is_entity_in_spans(e["ent"], spans)) for e in my_example.entities_with_labels]
present = [(e,idx) for e, idx in present_with_indices if idx is not None]
missing = [(e,idx) for e, idx in present_with_indices if idx is None]

# Optional: Create a mapping of entity to span index for easy lookup
#entity_to_span_idx = {e["ent"]: idx for e, idx in present_with_indices if idx is not None}
print(f"present_with_indices:\n\t{present_with_indices}")
print(f"present:\n\t{present}")
print(f"missing:\n\t{missing}")

*Block to Analyze a single example*

In [ ]:
# TODO:
# logic for counting occurence of each error type
# Intergrate logging template (dataclass)
# Create file error_categorization


def _check_entity_detection(
    example: BehaviouralExample,
    spans: List[Dict], # TODO: update to List[Spans]
)-> List[Dict] | List:
    all_errors = []
    missing_entities = []

    # 1) Match expected entities -> predicted span index (lenient substring match)
    present_with_indices = [(e, _is_entity_in_spans(e["ent"], spans)) for e in example.entities_with_labels]
    present = [(e, idx) for e, idx in present_with_indices if idx is not None]
    missing = [e for e, idx in present_with_indices if idx is None]

    matched_span_indices = {idx for _, idx in present}

    # 2) Missing entities (type_1)
    for e in missing:
        missing_entities.append({
            "entity": e,
            "error_type": ErrorTaxonomy.type_1,
            "reasoning": f"Expected entity '{e['ent']}' was not detected",
        })

    # 3) Errors for detected entities
    for entity_data, span_idx in present:
        span = spans[span_idx]

        expected_start, expected_end = entity_data["start"], entity_data["end"]
        predicted_start, predicted_end = span["start"], span["end"]
        predicted_label = span["label"]
        true_label = entity_data["label"]

        entity_errors = []

        label_errors = _check_predicted_labels(predicted_label=predicted_label, label=true_label)
        if label_errors:
            entity_errors.extend(label_errors if isinstance(label_errors, list) else [label_errors])

        boundary_errors = _check_boundaries(
            expected_start=expected_start,
            expected_end=expected_end,
            predicted_start=predicted_start,
            predicted_end=predicted_end,
        )
        if boundary_errors:
            entity_errors.append(boundary_errors)

        if entity_errors:
            all_errors.append({
                "entity": entity_data,
                "span": span,
                "span_idx": span_idx,
                "errors": entity_errors,
            })

    # 4) False positives: predicted entity spans not matched to any expected entity
    false_positives = []
    for span_idx, span in enumerate(spans):
        if span["label"] == "O":
            continue
        if span_idx in matched_span_indices:
            continue
        false_positives.append({
            "span": span,
            "span_idx": span_idx,
            "error_type": [ErrorTaxonomy.type_0,ErrorTaxonomy.type_7],  # later we will worry about differentiating these cases
            "reasoning": f"Predicted entity span '{span['text']}' not in expected entities",
        })

    # --- Example-level status summary (safe; never calls len(None)) ---
    # Summarize boundary vs label issues by scanning the collected per-entity errors.
    boundary_issue_types = {ErrorTaxonomy.type_5, ErrorTaxonomy.type_6}
    label_issue_types = {ErrorTaxonomy.type_1, ErrorTaxonomy.type_2, ErrorTaxonomy.type_3, ErrorTaxonomy.type_4}

    boundary_errors = [
        err
        for item in all_errors
        for err in item.get("errors", [])
        if err in boundary_issue_types
    ]
    label_errors = [
        err
        for item in all_errors
        for err in item.get("errors", [])
        if err in label_issue_types
    ]

    boundary_errors_count = len(boundary_errors)
    label_errors_count = len(label_errors)
    false_positives_count = len(false_positives)
    missing_entities_count = len(missing_entities)

    status_strings = []
    if boundary_errors_count:
        status_strings.append(f"   🟥 Boundary errors: {boundary_errors_count}")
    if label_errors_count:
        status_strings.append(f"   🟧 Label errors: {label_errors_count}")
    if false_positives_count:
        status_strings.append(f"   🟦 False positives: {false_positives_count}")
    if missing_entities_count:
        status_strings.append(f"   🟨 Missing entities: {missing_entities_count}")

    issues_exist = any(
        [boundary_errors_count, label_errors_count, false_positives_count, missing_entities_count]
    )

    print(
        "\n🔍 Example Status:\n"
        + ("\n".join(status_strings) + "\n" if status_strings else "")
        + ("🌟 No issues detected!" if not issues_exist else "⚠️ Issues detected!")
    )

    return {
        "boundary_errors": boundary_errors,
        "label_errors": label_errors,
        "present_errors": all_errors,  # boundary+label errors per expected entity span
        "missing_entities": missing_entities,
        "false_positives": false_positives,
    }



In [ ]:
# Evaluate every behavioural example and store results per theme
# NOTE: dict.fromkeys(..., []) would reuse the SAME list for every key; use a dict-comprehension instead.
errors_dict = {theme: [] for theme in behavioural_set.keys()}

for theme, examples in behavioural_set.items():
    print(f"\nTHEME: {theme} ({len(examples)} examples)")

    for example in examples:
        text = example.example
        tokens, token_labels, word_ids, words, word_labels, word_offsets = predict_word_level(
            text=text,
            model=model,
            tokenizer=tokenizer,
            id2label=id2label,
            device=device,
        )
        spans = word_labels_to_spans(text=text,
        word_offsets=word_offsets,
        word_labels=word_labels)

        errors = _check_entity_detection(
            example=example,
            spans=spans,
        )
        errors_dict[theme].append(errors)

# `errors_dict[theme]` is now a list of error summaries (one per example).



In [19]:
errors_dict

{'Core symptom mention (clean baseline)': [{'boundary_errors': [],
   'label_errors': [],
   'present_errors': [],
   'missing_entities': [],
   'false_positives': []},
  {'boundary_errors': [],
   'label_errors': [],
   'present_errors': [],
   'missing_entities': [],
   'false_positives': []},
  {'boundary_errors': [],
   'label_errors': ['Tokenization Artifacts', 'BIO Sequencing Errors'],
   'present_errors': [{'entity': {'ent': 'bradypnea',
      'label': 'SYMPTOM_POS',
      'start': 13,
      'end': 22},
     'span': {'start': 13,
      'end': 22,
      'text': 'bradypnea',
      'label': 'CONFLICT-B-SYMPTOM_NEG-I-SYMPTOM_POS-I-SYMPTOM_NEG-I-SYMPTOM_NEG'},
     'span_idx': 2,
     'errors': ['Tokenization Artifacts', 'BIO Sequencing Errors']}],
   'missing_entities': [],
   'false_positives': []},
  {'boundary_errors': [],
   'label_errors': ['Incorrect Polarity Assignment'],
   'present_errors': [{'entity': {'ent': 'dysphonia',
      'label': 'SYMPTOM_POS',
      'start': 13,
  

## **Build AI Evaluation Loop** - LATER 

In [ ]:
# from xai_sdk import Client
# from xai_sdk.chat import system, user

# from pydantic import BaseModel

# class DummyInfo(BaseModel):
#     message: str

# client = Client(api_key=os.getenv("XAI_API_KEY"))
# chat = client.chat.create(model="grok-4-1-fast-non-reasoning-latest")


# # test_prompt = "Say hello in a JSON object under the key 'message'."
# # chat.append(user(test_prompt))
# # response, invoice = chat.parse(DummyInfo)

In [ ]:
# # Extract error type keys from ErrorTaxonomy dataclass
# ErrorTypeKeys =ErrorTaxonomy.__dataclass_fields__.keys()
# # Create a set of valid error type keys for validation
# _valid_error_types = set(ErrorTypeKeys)

# # Create an Enum from the valid error types
# ErrorType = Enum('ErrorType', {k: k for k in _valid_error_types})

# class LLMResponse(BaseModel):
#     """
#     Response from LLM categorizing errors in NER predictions.
#     """
#     error_type: Optional[ErrorType] = Field(
#         default=None,
#         description=f"The error category code. Must be one of: {_valid_error_types}"
#     )
#     reasoning: str = Field(
#         description="Explanation of the error categorization, including which spans/entities are problematic and why"
#     )
    

In [ ]:
# SYSTEM_PROMPT = """You are an expert at analyzing Named Entity Recognition (NER) model predictions for clinical symptom extraction.
# Your task is to categorize errors in model predictions by comparing predicted spans against expected entities.
# SPAN Entities: SYMPTOM_POS (a symptom reported by the patient.), SYMPTOM_NEG (a symptom negated by the patient.), O, not an entity

# ## Error Taxonomy
# You must classify errors into ONE of these 8 categories (or return None if no error):

# IMPORTANT: Return the SHORT CODE (type_0, type_1, etc.) NOT the full name!

# - type_0: Irrelevant Span Mislabeling
#   - A non-relevant span is incorrectly detected as an entity
#   - Example: `65` labeled as SYMPTOM_POS
#   - Indicates semantic confusion—model cannot distinguish between entity and non-entity tokens

# - type_1: Missing Expected Entity
#   - A clinically relevant symptom is not detected at all
#   - Example: `exanthema` not extracted
#   - Strong signal for dataset enrichment

# - type_2: Tokenization Artifacts
#   - Errors caused by subword splits influencing predictions
#   - Example: Mixed labels across subwords of a single word, conflicting labels aggregated into `CONFLICT-*`
#   - Expected behavior with WordPiece/BPE tokenizers

# - type_3: Incorrect Polarity Assignment
#   - Incorrect polarity assignment in the presence of negation
#   - Example: `muscle cramps` labeled as `SYMPTOM_NEG` when context implies presence
#   - Model struggles with negation scope and contrastive clauses

# - type_4: BIO Sequencing Errors
#   - Incorrect or inconsistent BIO tag transitions
#   - Example: `I-SYMPTOM_POS` without a preceding `B-`, new symptom starting with `I-` instead of `B-`
#   - Model uncertainty at entity boundaries

# - type_5: Boundary Overreach
#   - Model captures a symptom PLUS unrelated surrounding words
#   - Example: `blisters on his head` (should be just `blisters`)
#   - Core entity detected correctly but span precision is low

# - type_6: Boundary Undereach
#   - Model captures only part of a multi-word symptom
#   - Example: `chest` `pain` instead of `chest pain`
#   - Partial entity detection

# - type_7: Model Overgeneralization
#   - Model predicts a symptom where none exists
#   - Example: Predicting `SYMPTOM_POS` for person names (`Marie`), general states (`well`), demographics (`65`)
#   - Model has learned overly broad symptom cues

# ## Input Format

# You will receive:
# - `input_text`: Original text
# - `tokens`: Tokenized tokens (filtered, no special tokens)
# - `token_level_labels`: BIO labels for each token (e.g., "B-SYMPTOM_POS", "I-SYMPTOM_POS", "O")
# - `word_ids`: Word index for each token (tokens from same word share same word_id)
# - `words`: Naive word split of text
# - `word_level_labels`: Aggregated BIO labels per word
# - `predicted_spans`: List of predicted entity spans with {start, end, text, label}
# - `expected_entities`: List of expected entity text strings (from ground truth)


# ## Your Task

# 1. Compare predicted_spans against expected_entities
# 2. Identify discrepancies (missing entities, extra entities, wrong boundaries, wrong polarity)
# 3. Classify the PRIMARY error type (choose the most significant issue)
# 4. Return the SHORT CODE (type_0, type_1, type_2, etc.) in the error_type field
# 5. Provide clear reasoning explaining:
#    - Which spans/entities are problematic
#    - Why this error type was chosen
#    - What the model got right vs wrong
#    - No more than 30 words per explanation. Be concise.


# - If prediction matches expected perfectly, return error_type=None
# - If multiple error types apply, note all errors
# - CRITICAL: Use the exact short code format (type_0, type_1, etc.) - do NOT use full names
# - Be specific in reasoning—reference actual spans and text
# - Consider tokenization artifacts when spans don't align perfectly
# - Pay attention to BIO tag sequences and polarity (POS vs NEG)"""